In [3]:
import pandas as pd
import numpy as np

# ====== only change these two ======
N_UP = 3
HOLD_DAYS = 1
# ===================================

df = pd.read_csv("SPY_prices.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

df["up"] = df["Close"].diff() > 0
df["signal_buy"] = df["up"].rolling(window=N_UP).sum().eq(N_UP)

df["entry_price"] = df["Close"]
df["exit_price"] = df["Close"].shift(-HOLD_DAYS)

signal_idx = df.index[df["signal_buy"]].to_list()

taken = []
blocked_until = -1
for i in signal_idx:
    if i <= blocked_until:
        continue
    if pd.isna(df.loc[i, "exit_price"]):
        continue
    taken.append(i)
    blocked_until = i + HOLD_DAYS

trades = df.loc[taken, ["Date", "entry_price", "exit_price"]].copy()
trades = trades.rename(columns={"Date": "EntryDate"})

trades["Contribution"] = 1.0
trades["Shares"] = trades["Contribution"] / trades["entry_price"]
trades["ValueAtExit"] = trades["Shares"] * trades["exit_price"]
trades["Profit"] = trades["ValueAtExit"] - trades["Contribution"]

trades["ExitDate"] = df.loc[[i + HOLD_DAYS for i in taken], "Date"].values

# ---- mean profit (per $1 trade) ----
mean_profit = trades["Profit"].mean()  # pandas Series.mean() [web:173]
print(f"Mean profit per trade ($): {mean_profit:.6f}")


trades.head()



Mean profit per trade ($): 0.000031


,EntryDate,entry_price,exit_price,Contribution,Shares,ValueAtExit,Profit,ExitDate
3,1993-02-03,44.81250,45.00000,1.0,0.022315,1.004184,0.004184,1993-02-04
19,1993-02-26,44.40625,44.28125,1.0,0.022519,0.997185,-0.002815,1993-03-01
64,1993-05-03,44.31250,44.46875,1.0,0.022567,1.003526,0.003526,1993-05-04
66,1993-05-05,44.59375,44.43750,1.0,0.022425,0.996496,-0.003504,1993-05-06
81,1993-05-26,45.59375,45.43750,1.0,0.021933,0.996573,-0.003427,1993-05-27
